# Answer Key to the Data Wrangling with DataFrames Coding Quiz

Helpful resources:
http://spark.apache.org/docs/latest/api/python/pyspark.sql.html

In [1]:
%use spark

import org.apache.spark.sql.types.DataTypes
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.api.java.UDF1

In [2]:
// 1) import any other libraries you might need
// 2) instantiate a Spark session 
// 3) read in the data set located at the path "data/sparkify_log_small.json"
// 4) write code to answer the quiz questions 

var df = spark.read().json("data/sparkify_log_small.json")

# Question 1

Which page did user id "" (empty string) NOT visit?

In [3]:
df.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [4]:
// filter for users with blank user id
var blankPages = df.filter("userId == ''").select("page").alias("blank_pages").dropDuplicates()

// get a list of possible pages that could be visited
var allPages = df.select("page").dropDuplicates()

// find values in all_pages that are not in blank_pages
// these are the pages that the blank user did not go to
(allPages.collectAsList().toSet() subtract blankPages.collectAsList().toSet()).forEach({it -> println(it[0])})

Submit Downgrade
Downgrade
Logout
Save Settings
Settings
NextSong
Upgrade
Error
Submit Upgrade


# Question 2 - Reflect

What type of user does the empty string user id most likely refer to?


Perhaps it represents users who have not signed up yet or who are signed out and are about to log in.

# Question 3

How many female users do we have in the data set?

In [5]:
df.filter("gender == 'F'").select("userId", "gender").dropDuplicates().count()

462

# Question 4

How many songs were played from the most played artist?

In [6]:
df.filter("page == 'NextSong'")
    .select("Artist")
    .groupBy("Artist")
    .agg(mapOf("Artist" to "count"))
    .withColumnRenamed("count(Artist)", "Artistcount")
    .sort(desc("Artistcount"))
    .show(1)

+--------+-----------+
|  Artist|Artistcount|
+--------+-----------+
|Coldplay|         83|
+--------+-----------+
only showing top 1 row



# Question 5 (challenge)

How many songs do users listen to on average between visiting our home page? Please round your answer to the closest integer.



In [7]:
// TODO: filter out 0 sum and max sum to get more exact answer

val isHome = UDF1<String, Int> { page -> if (page == "Home") 1 else 0 }
spark.udf().register("isHome", isHome, DataTypes.IntegerType)

val userWindow = Window
    .partitionBy("userID")
    .orderBy(desc("ts"))
    .rangeBetween(Window.unboundedPreceding(), 0)

val cusum = df.filter("page == 'NextSong' or page == 'Home'")
    .select("userID", "page", "ts")
    .withColumn("homevisit", callUDF("isHome", df.col("page")))
    .withColumn("period", sum("homevisit").over(userWindow))

cusum.filter("page == 'NextSong'")
    .groupBy("userID", "period")
    .agg(mapOf("period" to "count"))
    .agg(mapOf("count(period)" to "avg"))
    .show()

+------------------+
|avg(count(period))|
+------------------+
| 6.898347107438017|
+------------------+

